In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error

from tensorflow.keras.layers import Input, Dense, LeakyReLU, Concatenate, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
# Seed setup for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Weighted "bce" functions
def weighted_bce_linear(y_true, y_pred, mask, weight_factor=10.0):
    """Linear scaling of BCE with heavier weighting on missing entries."""
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = 1.0 + (weight_factor - 1.0)*(1 - mask)
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)

def weighted_bce_base(y_true, y_pred, mask, base=2.0):
    """Base weighting for missing entries vs. observed entries."""
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = 1.0 + (base - 1.0)*(1 - mask)
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)

def weighted_bce_missing_only(y_true, y_pred, mask, base=0):
    """
    Only penalize missing entries:
      - Observed = weight 0
      - Missing  = weight 1
    """
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = 1 - mask
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)

# Dictionary that indicates which extra parameter each weighting method needs.
weighting_methods = {
    'linear': (weighted_bce_linear, 'weight_factor'),   # uses weight_factor
    'base': (weighted_bce_base, 'base'),                # uses base
    'missing_only': (weighted_bce_missing_only, None)   # no extra param needed
}

In [ ]:
def build_models(dim):
    """
    Build generator and discriminator in a GAIN-like architecture.
    - Generator receives (X, Z, M)
    - Discriminator receives (X_hat, M)
    """
    # --- Generator ---
    X_input = Input(shape=(dim,))
    Z_input = Input(shape=(dim,))
    M_input = Input(shape=(dim,))

    G_input = Concatenate()([X_input, Z_input, M_input])
    G_h = Dense(128)(G_input)
    G_h = LeakyReLU(alpha=0.2)(G_h)
    G_h = Dense(128)(G_h)
    G_h = LeakyReLU(alpha=0.2)(G_h)
    G_out = Dense(dim, activation='sigmoid')(G_h)
    # Combine observed parts (X) and generated parts (G_out)
    X_hat = Multiply()([M_input, X_input]) + Multiply()([(1 - M_input), G_out])
    generator = Model([X_input, Z_input, M_input], X_hat)

    # --- Discriminator ---
    X_hat_input = Input(shape=(dim,))
    D_input = Concatenate()([X_hat_input, M_input])
    D_h = Dense(128)(D_input)
    D_h = LeakyReLU(alpha=0.2)(D_h)
    D_h = Dense(128)(D_h)
    D_h = LeakyReLU(alpha=0.2)(D_h)
    D_out = Dense(dim, activation='sigmoid')(D_h)
    discriminator = Model([X_hat_input, M_input], D_out)

    return generator, discriminator

In [ ]:
datasets_path = '../datasets/datasets-by-source'
dataset_file = f'{datasets_path}/df_ap.csv'

# Hyperparams for searching
methods_to_test = ['linear']
weight_factors = [2.0, 5.0, 10.0]
bases = [2.0, 5.0, 10.0]

# Number of epochs in the sense of "full passes over the train data"
epochs_list = [500, 1000, 2000]
batchs_size_list = [64, 128]

# We'll store results in this list and then export to CSV
results = []

df = pd.read_csv(dataset_file)
# Drop columns not needed
# df = df.drop(columns=['Timestamp_cubic', 'Hop_count', 'Bottleneck', 'Link_bottleneck'])

df = df.drop(columns=[  'Link_bottleneck'])

# Split data
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
X_train_full = train_df
X_test_full = test_df

# column_names = ["Vazao", "Vazao_bbr", 'Timestamp_cubic', 'Hop_count', 'Bottleneck']
# X_test_full = pd.DataFrame(X_test_full, columns=column_names)

In [ ]:
# ---------------------------------
# No missing data in the training set
# ---------------------------------
# X_train_full: shape (n_train, dim)
# M_train: mask for real missingness (if any), else all ones
M_train_missing = np.isnan(X_train_full).astype(float)
M_train = 1.0 - M_train_missing

# Mean imputation for training to compute normalization stats
imputer_train_mean = SimpleImputer(strategy='mean')
X_train_imputed_mean = imputer_train_mean.fit_transform(X_train_full)

# Compute min/max for later normalization
X_min = np.nanmin(X_train_imputed_mean, axis=0)
X_max = np.nanmax(X_train_imputed_mean, axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1  # avoid division by zero

# Normalize training data
X_train_norm = (X_train_imputed_mean - X_min) / X_range
dim = X_train_full.shape[1]

# ---------------------------------
# 10% missing in test set (columns "Vazao" and "Vazao_bbr")
# ---------------------------------
# Suppose X_test_full is a pandas DataFrame with columns:
# "Vazao", "Vazao_bbr", and others
col_vazao = X_test_full.columns.get_loc("Vazao")
col_vazao_bbr = X_test_full.columns.get_loc("Vazao_bbr")

n_test = X_test_full.shape[0]
num_missing_vazao = int(n_test * 0.10)
num_missing_vazao_bbr = int(n_test * 0.10)

# Randomly pick rows to set missing
missing_rows_vazao = np.random.choice(n_test, num_missing_vazao, replace=False)
missing_rows_vazao_bbr = np.random.choice(n_test, num_missing_vazao_bbr, replace=False)

# Make a copy and set those positions to NaN
X_test_missing = X_test_full.copy()
X_test_missing.iloc[missing_rows_vazao, col_vazao] = np.nan
X_test_missing.iloc[missing_rows_vazao_bbr, col_vazao_bbr] = np.nan

# Observed/missing mask
M_test_missing = X_test_missing.isna().astype(float)  # 1 if missing
M_test = 1.0 - M_test_missing                         # 1 if observed

# Mean-impute test to get a base for normalization
imputer_test_mean = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer_test_mean.fit_transform(X_test_missing)
X_test_norm = (X_test_imputed_mean - X_min) / X_range

# Flatten the original test for "true" missing values
X_test_flat = X_test_full.values.flatten()

# Build a list of flatten indices that are artificially missing
test_missing_positions_vazao = [i * dim + col_vazao for i in missing_rows_vazao]
test_missing_positions_bbr = [i * dim + col_vazao_bbr for i in missing_rows_vazao_bbr]
test_missing_positions = np.array(test_missing_positions_vazao + test_missing_positions_bbr, dtype=int)

# Get the "true" values for those missing positions
Y_true_missing_test_vazao = X_test_full.values[missing_rows_vazao, col_vazao]
Y_true_missing_test_bbr = X_test_full.values[missing_rows_vazao_bbr, col_vazao_bbr]
Y_true_missing_test = np.concatenate([Y_true_missing_test_vazao, Y_true_missing_test_bbr])

# ------------------------------------------------
# (1) Mean Imputation
# ------------------------------------------------
imputer_mean = SimpleImputer(strategy='mean')
X_test_imputed_mean = imputer_mean.fit_transform(X_test_missing)

# Flatten to index the same positions
X_mean_flat = X_test_imputed_mean.flatten()

# Extract predicted values for artificially missing entries
Y_pred_missing_mean = X_mean_flat[test_missing_positions]

# Compute MSE and then RMSE (old style if 'squared' param not available)
mse_mean = mean_squared_error(Y_true_missing_test, Y_pred_missing_mean)
rmse_mean = np.sqrt(mse_mean)

print("Mean Imputation - MSE:", mse_mean, "RMSE:", rmse_mean)


# ------------------------------------------------
# (2) KNN Imputation
# ------------------------------------------------
imputer_knn = KNNImputer(n_neighbors=5)
X_test_imputed_knn = imputer_knn.fit_transform(X_test_missing)

# Flatten to index the same positions
X_knn_flat = X_test_imputed_knn.flatten()

# Extract predicted values for artificially missing entries
Y_pred_missing_knn = X_knn_flat[test_missing_positions]

# Compute MSE and then RMSE
mse_knn = mean_squared_error(Y_true_missing_test, Y_pred_missing_knn)
rmse_knn = np.sqrt(mse_knn)

print("KNN Imputation - MSE:", mse_knn, "RMSE:", rmse_knn)



In [ ]:
# ================================================
# 4.4 HYPERPARAM SEARCH LOOP (ADJUSTED)
# ================================================

best_rmse = np.inf
best_params = None

for method_name in methods_to_test:
    w_func, param_name = weighting_methods[method_name]

    if param_name == 'weight_factor':
        param_values = weight_factors
    elif param_name == 'base':
        param_values = bases
    else:
        param_values = [None]

    for param_val in param_values:
        for n_epochs in epochs_list:
            for batch_size in batchs_size_list:

                if param_name:
                    print(f"\nTesting weighted method={method_name}, {param_name}={param_val}, epochs={n_epochs}")
                else:
                    print(f"\nTesting weighted method={method_name}, epochs={n_epochs}")

                # Build models & optimizers
                generator, discriminator = build_models(dim)
                d_optimizer = Adam(learning_rate=0.0002, beta_1=0.5)
                g_optimizer = Adam(learning_rate=0.0002, beta_1=0.5)

                # --- Training loop parameters ---
                X_data = X_train_norm
                M_data = M_train
                n_samples = X_data.shape[0]
                steps_per_epoch = n_samples // batch_size

                # --------------------------------------------------
                # 4.4.1 TRAIN FOR n_epochs
                # --------------------------------------------------
                for epoch in range(1, n_epochs + 1):
                    for step in range(steps_per_epoch):
                        idx = np.random.randint(0, n_samples, batch_size)

                        # Convert arrays to Tensors
                        X_batch_tf = tf.convert_to_tensor(X_data[idx], dtype=tf.float32)
                        M_batch_tf = tf.convert_to_tensor(M_data.iloc[idx], dtype=tf.float32)

                        # Random Z
                        Z_batch_tf = tf.random.uniform(
                            shape=(batch_size, dim),
                            minval=0.,
                            maxval=1.,
                            dtype=tf.float32
                        )

                        # --- Discriminator update ---
                        with tf.GradientTape() as tape_d:
                            X_hat_batch = generator([X_batch_tf, Z_batch_tf, M_batch_tf], training=True)
                            D_labels = M_batch_tf  # using M as "real" labels
                            D_pred_out = discriminator([X_hat_batch, M_batch_tf], training=True)

                            # Weighted BCE
                            if param_name == 'weight_factor':
                                d_loss = w_func(D_labels, D_pred_out, M_batch_tf, weight_factor=param_val)
                            elif param_name == 'base':
                                d_loss = w_func(D_labels, D_pred_out, M_batch_tf, base=param_val)
                            else:
                                d_loss = w_func(D_labels, D_pred_out, M_batch_tf)

                        d_grads = tape_d.gradient(d_loss, discriminator.trainable_variables)
                        d_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

                        # --- Generator update ---
                        with tf.GradientTape() as tape_g:
                            X_hat_g = generator([X_batch_tf, Z_batch_tf, M_batch_tf], training=True)
                            G_pred_out = discriminator([X_hat_g, M_batch_tf], training=True)

                            # "Trick" labels are all ones
                            trick_labels_tf = tf.ones(shape=(batch_size, dim), dtype=tf.float32)
                            g_loss = tf.keras.losses.binary_crossentropy(trick_labels_tf, G_pred_out)
                            g_loss = tf.reduce_mean(g_loss)

                        g_grads = tape_g.gradient(g_loss, generator.trainable_variables)
                        g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

                    # Print progress
                    if epoch % 200 == 0 or epoch == n_epochs:
                        print(f"  Epoch [{epoch}/{n_epochs}] d_loss={d_loss:.4f}, g_loss={g_loss:.4f}")

                # --------------------------------------------------
                # 4.4.2 EVALUATE ON TEST AFTER TRAINING
                # --------------------------------------------------
                X_test_tf = tf.convert_to_tensor(X_test_norm, dtype=tf.float32)
                M_test_tf = tf.convert_to_tensor(M_test.values, dtype=tf.float32)  # or M_test if NumPy
                Z_full_test_tf = tf.random.uniform(
                    shape=(X_test_tf.shape[0], dim),
                    minval=0.,
                    maxval=1.,
                    dtype=tf.float32
                )

                # Impute with generator
                X_imputed_norm = generator([X_test_tf, Z_full_test_tf, M_test_tf], training=False)
                X_imputed_final = X_imputed_norm.numpy() * X_range + X_min

                # Compare only the artificially-missing positions
                X_gan_flat = X_imputed_final.flatten()
                Y_pred_missing_gan = X_gan_flat[test_missing_positions]

                # rmse_gan_imputation = mean_squared_error(Y_true_missing_test, Y_pred_missing_gan,
                #                                          squared=False)  # squared=False for RMSE
                
                mse = mean_squared_error(Y_true_missing_test, Y_pred_missing_gan)  # old version
                rmse_gan_imputation = np.sqrt(mse)

                print(f"  => RMSE on test: {rmse_gan_imputation:.6f} "
                      f"(method={method_name}, {param_name}={param_val}, epochs={n_epochs})")

                # Track best
                if rmse_gan_imputation < best_rmse:
                    best_rmse = rmse_gan_imputation
                    best_params = (method_name, param_name, param_val, n_epochs)

# Finally, after the loop
print("\nBest RMSE:", best_rmse)
print("Best hyperparams:", best_params)


In [ ]:
# # =================================================
# # 5. PRINT & SAVE RESULTS
# # =================================================
# print("\n" + f"Source of these results: df_ap.csv")
# print("Best result for GAN:")
# if best_params and best_params[1]:
#     print(f"RMSE: {best_rmse}, method={best_params[0]}, "
#           f"{best_params[1]}={best_params[2]}, epochs={best_params[3]}")
# elif best_params:
#     print(f"RMSE: {best_rmse}, method={best_params[0]}, epochs={best_params[3]}")
# else:
#     print("No best_params found (no training run?).")

# print("\nRMSE Comparison (Test Set):")
# print(f"Mean: {rmse_mean_imputation}")
# print(f"KNN : {rmse_knn_imputation}")
# print(f"GAN (best found): {best_rmse}")
# print("-"*60)

# # We’ll store in a list and then export to CSV
# results.append({
#     'file name': 'df_ap.csv',
#     'best gan rmse': best_rmse,
#     'knn rmse': rmse_knn_imputation,
#     'average rmse': rmse_mean_imputation
# })

# results_df = pd.DataFrame(results)
# results_df.to_csv('gan_imputation_results.csv', index=False)
# print("\nSaved results to 'gan_imputation_results.csv'.")
